# Optimización de modelos

## Objetivo del notebook

El objetivo de este notebook es optimizar los modelos de clasificación más prometedores obtenidos durante la fase de modelado mediante técnicas de búsqueda de hiperparámetros.

Para ello se utilizarán métodos de validación cruzada que permitan identificar la combinación de parámetros con mejor rendimiento y comparar posteriormente los modelos optimizados con el fin de seleccionar la mejor solución para la predicción del abandono de clientes.

# 1. Carga de librerías

In [1]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Optimización de modelos
from sklearn.model_selection import GridSearchCV

# Evaluación de modelos
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 2. Carga de datos

## Objetivo

En este apartado se cargan los conjuntos de entrenamiento y prueba generados durante el notebook de preprocesamiento.

Además, se prepara una versión numérica de la variable objetivo que permitirá entrenar todos los modelos utilizando un mismo flujo de trabajo durante la fase de optimización.

In [2]:
# Carga de los conjuntos de entrenamiento y prueba preprocesados

X_train = pd.read_csv("../data/processed/X_train_preprocesado.csv")
X_test = pd.read_csv("../data/processed/X_test_preprocesado.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze("columns")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze("columns")

In [3]:
# Convertir la variable objetivo a formato numérico

y_train = y_train.map({"No": 0, "Yes": 1})
y_test = y_test.map({"No": 0, "Yes": 1})

In [4]:
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

print("\nValores de la variable objetivo:")
print(y_train.value_counts())

X_train: (5634, 45)
X_test: (1409, 45)
y_train: (5634,)
y_test: (1409,)

Valores de la variable objetivo:
Abandono
0    4139
1    1495
Name: count, dtype: int64


# 3. Estrategia de optimización

## Objetivo

Una vez identificados los modelos con mejor rendimiento durante la fase de modelado, el siguiente paso consiste en optimizar sus hiperparámetros.

Para ello se utilizará **GridSearchCV**, una técnica que combina la búsqueda sistemática de distintas configuraciones de hiperparámetros con validación cruzada (*Cross Validation*). Este procedimiento permite seleccionar la combinación que ofrece el mejor rendimiento medio sobre diferentes particiones del conjunto de entrenamiento, reduciendo el riesgo de sobreajuste y proporcionando una estimación más robusta del comportamiento del modelo.

# 4. Optimización de la Regresión Logística

## Objetivo

La Regresión Logística fue el modelo que obtuvo el mejor rendimiento durante la comparación inicial de algoritmos.

En este apartado se optimizarán sus principales hiperparámetros utilizando **GridSearchCV** con validación cruzada, con el objetivo de identificar la configuración que proporcione el mejor rendimiento medio sobre el conjunto de entrenamiento.

In [9]:
# Hiperparámetros a evaluar

param_grid_logistic = {
    "C": [0.01, 0.1, 1, 10, 100]
}

### Configuración de GridSearchCV

Con el fin de identificar la mejor combinación de hiperparámetros, se emplea **GridSearchCV** utilizando validación cruzada de cinco particiones (*5-fold Cross Validation*).

Como métrica de evaluación se utilizará **ROC-AUC**, ya que proporciona una medida robusta de la capacidad del modelo para distinguir entre ambas clases y resulta especialmente adecuada en problemas de clasificación con clases desbalanceadas.

In [10]:
# Configuración de GridSearchCV

grid_logistic = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid_logistic,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

In [11]:
# Búsqueda de los mejores hiperparámetros

grid_logistic.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with hig

In [17]:
print("Mejores hiperparámetros:")
print(grid_logistic.best_params_)

Mejores hiperparámetros:
{'C': 100}


In [18]:
print(f"Mejor ROC-AUC medio: {grid_logistic.best_score_:.4f}")

Mejor ROC-AUC medio: 0.8460


### Evaluación del modelo optimizado

Una vez identificada la mejor combinación de hiperparámetros mediante validación cruzada, se evalúa el modelo optimizado sobre el conjunto de prueba.

Esta evaluación permite comprobar si el rendimiento obtenido durante la fase de optimización se mantiene cuando el modelo se enfrenta a datos no utilizados durante el entrenamiento.

In [19]:
# Mejor modelo encontrado

best_logistic = grid_logistic.best_estimator_

In [20]:
# Predicciones

y_pred = best_logistic.predict(X_test)
y_prob = best_logistic.predict_proba(X_test)[:, 1]

In [21]:
# Evaluación del modelo optimizado

resultados_logistic_opt = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

pd.DataFrame(
    resultados_logistic_opt,
    index=["Logistic Regression Optimizada"]
).round(4)

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Logistic Regression Optimizada,0.7999,0.6438,0.5508,0.5937,0.8409


### Interpretación

La optimización mediante **GridSearchCV** identificó como mejor configuración un valor de **C = 100**, obteniendo un **ROC-AUC medio de 0.8460** durante la validación cruzada.

Sin embargo, al evaluar el modelo optimizado sobre el conjunto de prueba, el rendimiento fue ligeramente inferior al obtenido por la configuración base utilizada en el notebook anterior.

Este resultado pone de manifiesto que una mejora durante el proceso de validación cruzada no garantiza necesariamente un mejor comportamiento sobre datos no vistos. Por este motivo, la evaluación final siempre debe realizarse utilizando el conjunto de prueba, que proporciona una estimación más realista de la capacidad de generalización del modelo.

# 5. Optimización de Random Forest

## Objetivo

Tras evaluar el rendimiento del modelo base, se optimizarán los principales hiperparámetros de Random Forest mediante **GridSearchCV**.

El objetivo es determinar si una configuración optimizada permite mejorar la capacidad predictiva del modelo respecto a la configuración utilizada durante la fase de comparación inicial.

In [22]:
# Hiperparámetros a evaluar

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [23]:
# Configuración de GridSearchCV

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

In [24]:
# Búsqueda de los mejores hiperparámetros

grid_rf.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose v

In [25]:
print("Mejores hiperparámetros:")
print(grid_rf.best_params_)

Mejores hiperparámetros:
{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}


In [26]:
print(f"Mejor ROC-AUC medio: {grid_rf.best_score_:.4f}")

Mejor ROC-AUC medio: 0.8451


### Evaluación del modelo optimizado

Una vez identificada la mejor combinación de hiperparámetros mediante validación cruzada, se evalúa el modelo optimizado sobre el conjunto de prueba para comprobar su capacidad de generalización sobre datos no utilizados durante el entrenamiento.

In [27]:
# Mejor modelo encontrado

best_rf = grid_rf.best_estimator_

In [28]:
# Predicciones

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

In [29]:
# Evaluación del modelo optimizado

resultados_rf_opt = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

pd.DataFrame(
    resultados_rf_opt,
    index=["Random Forest Optimizado"]
).round(4)

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Random Forest Optimizado,0.8048,0.6737,0.5134,0.5827,0.8378


### Interpretación

A diferencia de la Regresión Logística, la optimización de **Random Forest** sí produce una mejora consistente respecto a la configuración base.

El modelo optimizado obtiene mejores resultados en todas las métricas evaluadas, lo que indica que el ajuste de los hiperparámetros ha permitido mejorar su capacidad de generalización.

No obstante, al compararlo con la Regresión Logística optimizada, ambos modelos presentan un rendimiento muy similar. Mientras que Random Forest ofrece una mayor precisión y exactitud global, la Regresión Logística mantiene una ligera ventaja en Recall, F1-score y ROC-AUC.

Por este motivo, resulta necesario optimizar también **XGBoost** antes de seleccionar el modelo definitivo.

# 6. Optimización de XGBoost

## Objetivo

XGBoost es uno de los algoritmos de clasificación más potentes para problemas con datos tabulares. En este apartado se optimizarán sus principales hiperparámetros mediante **GridSearchCV** con el objetivo de comprobar si es posible mejorar el rendimiento obtenido con la configuración base evaluada en el notebook anterior.

Posteriormente, el modelo optimizado será comparado con la Regresión Logística y Random Forest optimizados para seleccionar el modelo final del proyecto.

In [31]:
# Hiperparámetros a evaluar

param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1]
}

In [32]:
# Configuración de GridSearchCV

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_grid=param_grid_xgb,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

In [33]:
# Búsqueda de los mejores hiperparámetros

grid_xgb.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.1], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Control

In [34]:
print("Mejores hiperparámetros:")
print(grid_xgb.best_params_)

Mejores hiperparámetros:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}


In [35]:
print(f"Mejor ROC-AUC medio: {grid_xgb.best_score_:.4f}")

Mejor ROC-AUC medio: 0.8468


In [36]:
best_xgb = grid_xgb.best_estimator_

In [37]:
# Predicciones

y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:, 1]

In [38]:
# Evaluación del modelo optimizado

resultados_xgb_opt = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

pd.DataFrame(
    resultados_xgb_opt,
    index=["XGBoost Optimizado"]
).round(4)

,Accuracy,Precision,Recall,F1-score,ROC-AUC
XGBoost Optimizado,0.8077,0.6758,0.5294,0.5937,0.8445


# 7. Comparación de modelos optimizados

## Objetivo

Una vez optimizados los tres modelos de clasificación, se comparan sus métricas de evaluación con el objetivo de seleccionar la solución que ofrece el mejor equilibrio entre capacidad predictiva y generalización sobre datos no vistos.

In [39]:
tabla_optimizados = pd.DataFrame(
    {
        "Logistic Regression": resultados_logistic_opt,
        "Random Forest": resultados_rf_opt,
        "XGBoost": resultados_xgb_opt
    }
).T

tabla_optimizados.round(4)

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Logistic Regression,0.7999,0.6438,0.5508,0.5937,0.8409
Random Forest,0.8048,0.6737,0.5134,0.5827,0.8378
XGBoost,0.8077,0.6758,0.5294,0.5937,0.8445


### Interpretación

Los resultados muestran que la optimización de hiperparámetros ha permitido mejorar el rendimiento de los modelos basados en árboles, especialmente en Random Forest y XGBoost.

Entre los modelos evaluados, **XGBoost** obtiene el mejor rendimiento global, alcanzando la mayor Accuracy, Precision y ROC-AUC, además de compartir el mejor F1-score con la Regresión Logística.

Sin embargo, la **Regresión Logística** presenta el mayor Recall, lo que significa que identifica un mayor porcentaje de clientes que realmente abandonan el servicio.

Por tanto, la elección del modelo no depende únicamente de las métricas, sino también del objetivo de negocio. Si la prioridad es maximizar la detección de clientes con riesgo de abandono, la Regresión Logística constituye una alternativa muy competitiva. Si se busca un equilibrio global entre todas las métricas de clasificación, XGBoost ofrece el mejor rendimiento.

# 8. Selección del modelo final

## Modelo seleccionado

La selección del modelo final debe realizarse considerando tanto el rendimiento estadístico como el objetivo de negocio.

Desde un punto de vista técnico, **XGBoost** obtiene el mejor rendimiento global tras la optimización de hiperparámetros, alcanzando los valores más elevados de Accuracy, Precision y ROC-AUC.

No obstante, dado que el objetivo de este proyecto es la **predicción del abandono de clientes**, la capacidad para identificar correctamente a los clientes con riesgo de abandono resulta especialmente relevante. En este aspecto, la **Regresión Logística** obtiene el mayor Recall, detectando un porcentaje ligeramente superior de clientes que finalmente abandonan el servicio.

Aunque ambos modelos presentan un rendimiento muy competitivo, **XGBoost se selecciona como modelo final del proyecto** por ofrecer el mejor equilibrio entre capacidad predictiva y generalización. No obstante, en un entorno empresarial donde la prioridad absoluta sea minimizar la pérdida de clientes, la Regresión Logística podría considerarse una alternativa igualmente válida debido a su mayor capacidad de detección.

# 9. Conclusiones

## Conclusiones

En este notebook se ha llevado a cabo la optimización de los tres modelos de clasificación con mejor rendimiento obtenidos durante la fase de modelado mediante **GridSearchCV** y validación cruzada.

Los resultados muestran que la optimización de hiperparámetros mejora especialmente el rendimiento de los modelos basados en árboles, mientras que la Regresión Logística mantiene un comportamiento muy competitivo con una configuración cercana a la utilizada inicialmente.

Desde un punto de vista global, **XGBoost** obtiene el mejor equilibrio entre Accuracy, Precision, F1-score y ROC-AUC, convirtiéndose en el modelo con mejor rendimiento general.

No obstante, dado que el objetivo del proyecto es predecir el abandono de clientes, el **Recall** adquiere una especial relevancia. En este aspecto, la Regresión Logística consigue detectar un porcentaje ligeramente superior de clientes que abandonan el servicio, lo que pone de manifiesto que la elección del modelo debe realizarse siempre considerando el contexto de negocio y no únicamente una métrica concreta.

En el siguiente notebook se realizará una evaluación detallada del modelo seleccionado, analizando su comportamiento mediante la matriz de confusión, la curva ROC y la importancia de las variables, con el objetivo de extraer conclusiones útiles para la toma de decisiones.